# Tests des fonctions de calcul de la saturation

In [ ]:
import geopandas as gpd
import pandas as pd
from pandas import NamedAgg
from datetime import date

from saturation import to_sampled_statuses, to_sampled_sessions, to_sampled_state_pdc, to_state_poc_d, to_sampled_state_grp, to_state_grp_h,  to_state_grp_d

## Test échantillonage des sessions

In [2]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [2.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]


test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=-1)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 2.1], [5.5, 7.5], [13.1, 15.1]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6]
# hp1 : [x, 1] [2.1, 5.5], [7.5, 13.1]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[5]['occupation_pdc'] == 'f_libre'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'
# res

## Test échantillonage des statuts

In [3]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
valeurs = [1, 1.2, 3, 3.5, 5, 6.1, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in valeurs],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'id_pdc_itinerance': pdc}) 
statuses = to_sampled_statuses(test, init, timestamp, echantillons)
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses

,periode,etat_pdc,id_pdc_itinerance
0,2025-04-25 00:00:00+02:00,en_service,p1
1,2025-04-25 01:00:00+02:00,en_service,p1
2,2025-04-25 02:00:00+02:00,en_service,p1
3,2025-04-25 03:00:00+02:00,en_service,p1
4,2025-04-25 04:00:00+02:00,en_service,p1
5,2025-04-25 05:00:00+02:00,en_service,p1
6,2025-04-25 06:00:00+02:00,en_service,p1
7,2025-04-25 07:00:00+02:00,en_service,p1
8,2025-04-25 08:00:00+02:00,en_service,p1
9,2025-04-25 09:00:00+02:00,en_service,p1


## Test assemblage des sessions et des statuts

In [4]:
sessions = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'], 
                       'periode': [0,1,2,0,1,2,0,1,2],
                       'occupation_pdc': ['occupe', 'f_libre', 'occupe', 'f_libre', 'occupe', 'f_libre','f_libre', 'occupe', 'f_libre']})
status = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p3', 'p3', 'p3', 'p4', 'p4', 'p4'], 
                       'periode': [0,1,2,0,1,2, 0,1,2],
                       'etat_pdc': ['hors_service', 'hors_service', 'en_service', 'en_service', 'hors_service', 'hors_service', 'en_service', 'hors_service', 'en_service']})
merged = pd.merge(sessions, status, how='outer', on=['id_pdc_itinerance', 'periode']).fillna('aaa')
merged

,id_pdc_itinerance,periode,occupation_pdc,etat_pdc
0,p1,0,occupe,hors_service
1,p1,1,f_libre,hors_service
2,p1,2,occupe,en_service
3,p2,0,f_libre,aaa
4,p2,1,occupe,aaa
5,p2,2,f_libre,aaa
6,p3,0,f_libre,en_service
7,p3,1,occupe,hors_service
8,p3,2,f_libre,hors_service
9,p4,0,aaa,en_service


In [5]:
merged = to_sampled_state_pdc(sessions, status)
assert list(merged['state'][0:4]) == ['occupe', 'hors_service', 'occupe', 'libre']
merged

,id_pdc_itinerance,periode,state
0,p1,0,occupe
1,p1,1,hors_service
2,p1,2,occupe
3,p2,0,libre
4,p2,1,occupe
5,p2,2,libre
6,p3,0,libre
7,p3,1,occupe
8,p3,2,hors_service
9,p4,0,libre


In [6]:
print(merged['state'])
merged['state'].str.replace('en_service', 'libre')
merged['state'] = merged['state'].str.replace('en_service', 'libre')
merged

0           occupe
1     hors_service
2           occupe
3            libre
4           occupe
5            libre
6            libre
7           occupe
8     hors_service
9            libre
10    hors_service
11           libre
Name: state, dtype: object


,id_pdc_itinerance,periode,state
0,p1,0,occupe
1,p1,1,hors_service
2,p1,2,occupe
3,p2,0,libre
4,p2,1,occupe
5,p2,2,libre
6,p3,0,libre
7,p3,1,occupe
8,p3,2,hors_service
9,p4,0,libre


In [7]:
merged_d = to_state_poc_d(merged, 3)
merged_d

,id_pdc_itinerance,occupe,hors_service,libre
0,p1,960.0,480.0,0.0
1,p2,480.0,0.0,960.0
2,p3,480.0,480.0,480.0
3,p4,0.0,480.0,960.0


## Test état global échantillonné d'un groupement de pdc

In [8]:
pourcent_sature = 0.1
pourcent_surcharge = 0.2
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'],
                     'periode' : [0, 1, 2, 0, 1, 2, 0, 1, 2],
                     'state' : ['occupe', 'hors_service', 'occupe', 'libre', 'occupe', 'libre', 'libre', 'occupe', 'hors_service']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3'],
                         'id_station_itinerance': ['s1', 's1', 's2']}) 
state_grp = to_sampled_state_grp(test, stations, 'id_station_itinerance', pourcent_sature, pourcent_surcharge)
assert list(state_grp['state']) == [3,5,3,2,5,1]
assert list(state_grp['surcharge']) == [False] * 6
assert list(state_grp['sature']) == [False, True, False] * 2
assert list(state_grp['hs']) == [False] * 5 + [True]
assert list(state_grp['actif']) == [True, False, True, False, False, False]
assert list(state_grp['inactif']) == [False, False, False, True, False, False]


state_grp

9 9


,id_station_itinerance,periode,occupe,hors_service,libre,nb_pdc,hs,inactif,sature,surcharge,actif,state
0,s1,0,1,0,1,2,False,False,False,False,True,3
1,s1,1,1,1,0,2,False,False,True,False,False,5
2,s1,2,1,0,1,2,False,False,False,False,True,3
3,s2,0,0,0,1,1,False,True,False,False,False,2
4,s2,1,1,0,0,1,False,False,True,False,False,5
5,s2,2,0,1,0,1,True,False,False,False,False,1


In [9]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2', 'p2', 'p2'],
                     'periode' : [0, 1, 2, 3, 4, 5,
                                  0, 1, 2, 3, 4, 5],
                     'state' : ['occupe', 'hors_service', 'occupe', 'occupe', 'hors_service', 'libre',
                                'libre', 'libre', 'occupe', 'hors_service', 'hors_service', 'libre']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2'],
                         'id_station_itinerance': ['s1', 's1']}) 
state_grp = to_sampled_state_grp(test, stations, 'id_station_itinerance', pourcent_sature, pourcent_surcharge)
assert list(state_grp['state']) == [3,2,5,5,1,2]

state_grp

12 12


,id_station_itinerance,periode,occupe,hors_service,libre,nb_pdc,hs,inactif,sature,surcharge,actif,state
0,s1,0,1,0,1,2,False,False,False,False,True,3
1,s1,1,0,1,1,2,False,True,False,False,False,2
2,s1,2,2,0,0,2,False,False,True,False,False,5
3,s1,3,1,1,0,2,False,False,True,False,False,5
4,s1,4,0,2,0,2,True,False,False,False,False,1
5,s1,5,0,0,2,2,False,True,False,False,False,2


In [10]:

state_grp['periode'] = pd.Timestamp("2025-01-20 15:00:00", tz="UTC") + pd.to_timedelta(state_grp['periode'] * 10, unit="min")
state_grp_h = to_state_grp_h(state_grp, 'id_station_itinerance', echantillons=144, duree_etat_min=75)
assert state_grp_h['hs'][0] == 10
assert state_grp_h['inactif'][0] == 20
assert state_grp_h['sature'][0] == 20
assert state_grp_h['surcharge'][0] == 0
assert state_grp_h['actif'][0] == 10
state_grp_h


,id_station_itinerance,periode,periode_h,nb_pdc,hs,inactif,sature,surcharge,actif,sature_h,surcharge_h,periode_iso,periode_paris
0,s1,2025-01-20,15,2,10.0,20.0,20.0,0.0,10.0,False,False,2025-01-20 15:00:00,2025-01-20 16:00:00+01:00


In [13]:

state_grp_test = pd.DataFrame({'id_station_itinerance': ['s1', 's1'],
                               'periode': date(2026, 1, 15),
                               'periode_h': [1,2],
                               'nb_pdc': 2,
                               'hs': [10, 15],
                               'inactif': [20, 30],
                               'sature': [20, 10],
                               'surcharge': [0, 5],
                               'actif': [10, 15],
                               'sature_h': [False, True],
                               'surcharge_h': [True, True],
                               })
grouped = state_grp_test.groupby(['id_station_itinerance', 'periode'])
grouped.agg(
    nb_pdc=NamedAgg("nb_pdc", "max"),
    nb_h=NamedAgg("periode", "count"),
    hs=NamedAgg("hs", "sum"),
    inactif=NamedAgg("inactif", "sum"),
    sature_cum=NamedAgg("sature", "sum"),
    sature_max=NamedAgg("sature", "max"),
    surcharge=NamedAgg("surcharge", "sum"),
    actif=NamedAgg("actif", "sum"),
    ).reset_index()

state_grp_d = to_state_grp_d(state_grp_test, 'id_station_itinerance')
assert state_grp_d['id_station_itinerance'][0] == "s1"
assert state_grp_d['periode'][0] == date(2026, 1, 15)
assert state_grp_d['nb_pdc'][0] == 2
assert state_grp_d['nb_h'][0] == 2
assert state_grp_d['hs'][0] == 25
assert state_grp_d['inactif'][0] == 50
assert state_grp_d['sature_cum'][0] == 30
assert state_grp_d['sature_max'][0] == 20
assert state_grp_d['surcharge'][0] == 5
assert state_grp_d['actif'][0] == 25

In [12]:
test = pd.DataFrame({'name':         ['hs', 'inactif', 'sature', 'surcharge', 'actif'],
                     'occupe':       [0, 0, 5, 3, 2],
                     'hors_service': [6, 2, 1, 2, 2],
                     'libre':        [0, 4, 0, 1, 2],
                     'nb_pdc':       [6, 6, 6, 6, 6]})
# 2e partie de la fonction : to_sampled_state_grp
test['hs'] = (test['libre'] + test['occupe'] == 0) & (test['hors_service'] > 0)
test['inactif'] = ~test['hs'] & (test['occupe'] == 0)
test['sature'] = ~test['hs'] & ~test['inactif'] & (test['libre']/test['nb_pdc'] < 0.1)
test['surcharge'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & (test['libre']/test['nb_pdc'] < 0.2)
test['actif'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & ~test['surcharge']
test['state'] = test['hs'] + test['inactif'] * 2 + test['actif'] * 3 + test['surcharge'] * 4 + test['sature'] * 5
test

,name,occupe,hors_service,libre,nb_pdc,hs,inactif,sature,surcharge,actif,state
0,hs,0,6,0,6,True,False,False,False,False,1
1,inactif,0,2,4,6,False,True,False,False,False,2
2,sature,5,1,0,6,False,False,True,False,False,5
3,surcharge,3,2,1,6,False,False,False,True,False,4
4,actif,2,2,2,6,False,False,False,False,True,3
